# Project 1 Module 1: Environment and PostGIS Practice

This workbook builds the Python and operational reasoning used to confirm an ETL project is running in the correct repository, Python environment, and database context.

Work through each part in order. Predict before running, complete each answer cell, then run its check. The exercises inspect configuration and build commands safely; they do not install packages, change the working directory, start or stop containers, expose secrets, or modify PostGIS.

## How to Use This Workbook

Predict first, complete each editable answer cell, then run its separate check. Unfinished TODOs may fail checks by design. All core work is read-only with respect to the repository, containers, and database.

In [1]:
from pathlib import Path
import importlib.metadata as metadata
import shutil
import subprocess
import sys
import yaml


def passed(exercise: str) -> None:
    print(f"PASS: {exercise}")


ROOT_MARKERS = ("environment.yml", "docker-compose.yml", "src")
print("Safe setup helpers ready.")

Safe setup helpers ready.


# Part 1: Paths and Parents

`Path` keeps path operations explicit. `.parent` moves one level upward without changing the process working directory.

## Exercise 1A

Create a path for `learning/starters/project1_module_1_environment_postgis_practice.ipynb`, then collect its filename, parent folder name, and grandparent folder name.

In [ ]:
practice_path = Path("learning/starters/project1_module_1_environment_postgis_practice.ipynb")
path_facts = {
    "filename": "",  # TODO
    "parent": "",  # TODO
    "grandparent": "",  # TODO
}
path_facts

In [ ]:
assert path_facts == {
    "filename": "project1_module_1_environment_postgis_practice.ipynb",
    "parent": "practice",
    "grandparent": "learning",
}, "Use .name and .parent to move through the learning/starters path."
passed("Exercise 1A")

# Part 2: Find the Project Root

A root marker is a file or directory expected only at the project root. A helper should inspect the starting path and each parent, then raise a clear error if no candidate contains every marker.

## Exercise 2A

Complete `find_project_root`. It must accept either a file or directory as its starting point and must not call `chdir`.

In [ ]:
def find_project_root(start: Path, markers: tuple[str, ...] = ROOT_MARKERS) -> Path:
    start = start.resolve()
    candidate = start.parent if start.is_file() else start
    for possible_root in (candidate, *candidate.parents):
        # TODO: return possible_root when every marker exists beneath it.
        pass
    raise FileNotFoundError(
        f"Could not find project root from {start}; expected markers: {', '.join(markers)}"
    )

In [ ]:
project_root = find_project_root(Path.cwd())
assert project_root.name == "calgary-spatial-etl"
assert all((project_root / marker).exists() for marker in ROOT_MARKERS)
try:
    find_project_root(Path("/tmp"), ("marker-that-should-not-exist",))
except FileNotFoundError as exc:
    assert "expected markers" in str(exc)
else:
    raise AssertionError("The helper must raise when markers are absent.")
passed("Exercise 2A")

# Part 3: Expected Files

A readiness check should report all missing paths together instead of failing at the first one.

## Exercise 3A

Build `expected_paths` from the current repository contracts, then compute relative names for anything missing.

In [ ]:
expected_relative = [
    "environment.yml",
    "docker-compose.yml",
    "sql/init.sql",
    "src/config.py",
    "src/extract.py",
    "src/transform.py",
]
expected_paths = []  # TODO: join every relative path to project_root.
missing_expected = []  # TODO: retain missing paths as project-relative strings.
missing_expected

In [ ]:
assert len(expected_paths) == len(expected_relative)
assert all(isinstance(path, Path) for path in expected_paths)
assert missing_expected == [], f"Missing project contracts: {missing_expected}"
passed("Exercise 3A")

# Part 4: Python and Package Context

`sys.executable` identifies the interpreter actually running this kernel. `sys.version_info` is structured version data. Package distribution versions answer a different question from successful imports: a package can be declared but missing, installed under a distribution name that differs from its import name, or importable from another location.

The repository's `environment.yml` requests Python 3.11 and geospatial/database libraries; its nested `pip` section adds `python-dotenv` and `pyyaml`.

In [ ]:
environment_spec = yaml.safe_load((Path.cwd() / "environment.yml").read_text(encoding="utf-8"))
requested_python = next(item for item in environment_spec["dependencies"] if str(item).startswith("python="))
package_versions = {
    distribution: metadata.version(distribution)
    for distribution in ("geopandas", "shapely", "pyproj", "PyYAML")
}
runtime_report = {
    "executable": sys.executable,
    "python": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
    "requested_python": requested_python,
    "packages": package_versions,
}
runtime_report

In [ ]:
assert runtime_report["requested_python"] == "python=3.11"
assert Path(runtime_report["executable"]).exists()
assert set(package_versions) == {"geopandas", "shapely", "pyproj", "PyYAML"}
assert environment_spec["name"] == "calgary-etl"
passed("Runtime context smoke check")

# Part 5: Docker Compose Vocabulary

A **service** is a configured workload; an **image** supplies its filesystem and software; a **port mapping** connects host and container ports; a named **volume** persists container data. Building `subprocess` arguments is safe because construction does not execute them.

## Exercise 5A

Read the Compose YAML and describe `postgis`. Then construct argument lists for `docker compose config` and `docker compose ps` without running either command.

In [ ]:
compose_spec = yaml.safe_load((Path.cwd() / "docker-compose.yml").read_text(encoding="utf-8"))
postgis_service = compose_spec["services"]["postgis"]
compose_facts = {
    "service": "",  # TODO
    "image": "",  # TODO
    "host_port": "",  # TODO
    "container_port": "",  # TODO
    "volume_name": "",  # TODO
}
config_command = []  # TODO: [program, subcommand, action]
status_command = []  # TODO: [program, subcommand, action]
compose_facts, config_command, status_command

In [ ]:
assert compose_facts == {
    "service": "postgis",
    "image": "postgis/postgis:15-3.4",
    "host_port": "5432",
    "container_port": "5432",
    "volume_name": "postgis_data",
}
assert config_command == ["docker", "compose", "config"]
assert status_command == ["docker", "compose", "ps"]
passed("Exercise 5A")

# Part 6: Interpret Status Without Controlling Containers

Sample output can be interpreted without contacting Docker. `running` means the container process is active; a published `5432` mapping says where a client would connect, not that database authentication or schema setup has succeeded.

## Debugging Drill

The parser below incorrectly treats every non-empty line as a running service. Fix it so only rows whose state is `running` are retained.

In [ ]:
sample_status = [
    {"Service": "postgis", "State": "running", "Publishers": [{"PublishedPort": 5432}]},
    {"Service": "worker", "State": "exited", "Publishers": []},
]
running_services = []
for row in sample_status:
    if row:  # BUG: replace this condition.
        running_services.append(row["Service"])
running_services

In [ ]:
assert running_services == ["postgis"]
assert sample_status[0]["Publishers"][0]["PublishedPort"] == 5432
passed("Status debugging drill")

# Part 7: SQL Initialization Statements

The repository's `sql/init.sql` switches to `calgary_gis`, then uses idempotent `CREATE EXTENSION IF NOT EXISTS` statements for `postgis` and `postgis_topology`. Reading and classifying these lines does not connect to a database.

## Exercise 7A

Extract only the extension names from the SQL text.

In [ ]:
sql_lines = [
    line.strip()
    for line in (Path.cwd() / "sql/init.sql").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
extension_names = []  # TODO: parse the final token before ';' from CREATE EXTENSION lines.
extension_names

In [ ]:
assert sql_lines[0] == r"\c calgary_gis;"
assert extension_names == ["postgis", "postgis_topology"]
assert all("IF NOT EXISTS" in line for line in sql_lines[1:])
passed("Exercise 7A")

# Part 8: Capstone Readiness Report

Combine repository, interpreter, package, Compose, and SQL evidence into a plain dictionary. The report must contain no environment-variable values or passwords and must not execute external commands.

A component is ready only when its local, inspectable evidence passes.

In [ ]:
def build_readiness_report(root: Path) -> dict[str, object]:
    # TODO: derive each Boolean from evidence collected above.
    return {
        "project_root": str(root),
        "expected_files_ready": False,
        "python_311_ready": False,
        "packages_ready": False,
        "compose_contract_ready": False,
        "sql_contract_ready": False,
        "external_status_checked": False,
    }


readiness_report = build_readiness_report(project_root)
readiness_report

In [ ]:
expected_keys = {
    "project_root", "expected_files_ready", "python_311_ready", "packages_ready",
    "compose_contract_ready", "sql_contract_ready", "external_status_checked",
}
assert set(readiness_report) == expected_keys
assert all(readiness_report[key] is True for key in expected_keys - {"project_root", "external_status_checked"})
assert readiness_report["external_status_checked"] is False
assert "password" not in repr(readiness_report).lower()
passed("Readiness capstone")

## Optional Read-Only Live Docker Status

This optional cell is not required by any check. It runs only `docker compose ps`, only when the Docker executable exists, and never starts, stops, or modifies containers.

In [ ]:
if shutil.which("docker"):
    result = subprocess.run(
        ["docker", "compose", "ps"],
        cwd=Path.cwd(),
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout or result.stderr)
else:
    print("Docker executable not available; optional status skipped.")

# Bridge Back: Explain the Context in Plain English

Complete this dictionary without code jargon. Explain what proves the repository is correct, what the interpreter path tells you, why an import and a declared dependency are different evidence, what a Compose port mapping means, and why `IF NOT EXISTS` matters.

In [ ]:
bridge_back = {
    "root_evidence": "",
    "interpreter_evidence": "",
    "declared_vs_importable": "",
    "port_mapping": "",
    "idempotent_extension_sql": "",
}
for topic, explanation in bridge_back.items():
    print(f"{topic}: {explanation or '[not answered]'}")

# Optional Reference Solutions

Open only after a genuine attempt.

<details>
<summary>Paths, root search, and expected files</summary>

```python
path_facts = {"filename": practice_path.name, "parent": practice_path.parent.name, "grandparent": practice_path.parent.parent.name}
if all((possible_root / marker).exists() for marker in markers):
    return possible_root
expected_paths = [project_root / relative for relative in expected_relative]
missing_expected = [str(path.relative_to(project_root)) for path in expected_paths if not path.exists()]
```
</details>

<details>
<summary>Compose, SQL, status, and capstone decisions</summary>

```python
port_pair = postgis_service["ports"][0].split(":")
compose_facts = {"service": "postgis", "image": postgis_service["image"], "host_port": port_pair[0], "container_port": port_pair[1], "volume_name": postgis_service["volumes"][0].split(":")[0]}
config_command = ["docker", "compose", "config"]
status_command = ["docker", "compose", "ps"]
if row["State"] == "running":
    running_services.append(row["Service"])
extension_names = [line.removesuffix(";").split()[-1] for line in sql_lines if line.startswith("CREATE EXTENSION")]
```

For the capstone, derive each flag with `not missing_expected`, the Python major/minor tuple, `all(package_versions.values())`, exact Compose facts, and exact extension names. Keep `external_status_checked` false because core readiness does not contact Docker.
</details>

# Suggested Review Schedule

- **Today:** Paths, root markers, and expected-file checks.
- **Tomorrow:** Rebuild the runtime and Compose reports without solutions.
- **In three days:** Fix the status drill and explain the SQL statements.
- **In one week:** Rebuild the readiness capstone from a blank function.
- **In two weeks:** Diagnose a deliberately wrong project path and explain every failed readiness flag.

## AI Practice Review

After completing the working copy, save it and ask Copilot:

> Review my completed practice notebook without assigning a progression grade. Preserve my original answers and code. Inspect my reasoning, implementations, self-checks, errors, and outputs. Cite evidence for demonstrated strengths and misconceptions, recommend the smallest useful exercises to retry, and ask targeted follow-up questions before giving complete corrected answers.

After reviewing the feedback, preserve the completed notebook with `python scripts/save_attempt.py 1`.